### 3 · Retrieval Strategies — CMS3 Log Debugging

This notebook compares retrieval strategies for the CMS3 debugging assistant using the Pinecone index we created in Notebook 2.

### Strategies tested
```
1. Semantic only        → plain vector similarity search
2. Customer scoped      → semantic search + metadata filters
3. Hybrid debug search  → summary chunks + event chunks
4. Comparison           → side-by-side on realistic questions
```

For this feature, retrieval quality is mostly about using the right business keys like `customer_id` and the right chunk types at the right time.


In [1]:
import json
import os
import re
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

# Load environment variables from the project root so the notebook works in Jupyter.
NOTEBOOK_DIR = Path.cwd()
load_dotenv(dotenv_path=NOTEBOOK_DIR.parent / ".env", override=True)

EMBEDDING_MODEL = "text-embedding-3-small"
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "move-mind-ai")
INDEX_HOST = os.getenv("PINECONE_INDEX_HOST")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "cms3-logs")

required_env = ["OPENAI_API_KEY", "PINECONE_API_KEY"]
missing = [key for key in required_env if not os.getenv(key)]
if missing:
    raise ValueError(f"Missing required environment variables: {missing}")

# Connect to the same Pinecone-backed vector store used by the app.
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(host=INDEX_HOST) if INDEX_HOST else pc.Index(INDEX_NAME)
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

print(f"Connected to Pinecone index: {INDEX_NAME}")
print(f"Namespace: {NAMESPACE}")
print(f"Host override: {INDEX_HOST or '<not set>'}")

/Users/sauravmajumdar/Developer/AI/move-mind-ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Pinecone index: move-mind-ai
Namespace: cms3-logs
Host override: https://move-mind-ai-g57210f.svc.aped-4627-b74a.pinecone.io


In [2]:
# Helper functions keep the retrieval experiments readable.
CID_PATTERN = re.compile(r"\bCID\s*[:#-]?\s*(\d+)\b", re.IGNORECASE)
PAGE_PATTERN = re.compile(r"(/[A-Za-z0-9._/-]+)")


def show_results(results, label="Results"):
    print(f"\n{label} ({len(results)} chunks)")
    print("-" * 100)
    for i, doc in enumerate(results, start=1):
        m = doc.metadata
        print(
            f"[{i}] type={m.get('chunk_type')} | "
            f"cid={m.get('customer_id')} | "
            f"exec={m.get('execution_id')} | "
            f"action={m.get('action')} | "
            f"page={m.get('page_path')}"
        )
        print(doc.page_content[:350])
        print()


def extract_customer_id(query: str) -> str | None:
    match = CID_PATTERN.search(query)
    return match.group(1) if match else None


def extract_page_path(query: str) -> str | None:
    match = PAGE_PATTERN.search(query)
    return match.group(1) if match else None


def build_scope_filter(query: str, *, chunk_type: str | None = None) -> dict | None:
    metadata_filter = {}

    # Customer ID is the main business key in this product, so use it first.
    customer_id = extract_customer_id(query)
    if customer_id:
        metadata_filter["customer_id"] = customer_id

    # Exact page filters help when the user asks about a specific route.
    page_path = extract_page_path(query)
    if page_path:
        metadata_filter["page_path"] = page_path

    if chunk_type:
        metadata_filter["chunk_type"] = chunk_type

    return metadata_filter or None


def dedupe_docs(docs):
    unique_docs = []
    seen = set()
    for doc in docs:
        key = (doc.page_content, json.dumps(doc.metadata, sort_keys=True, default=str))
        if key in seen:
            continue
        seen.add(key)
        unique_docs.append(doc)
    return unique_docs


TEST_QUERIES = [
    "For CID 7093495, what happened in this journey?",
    "For CID 7093495, which UI condition evaluated to true on /ccflownew/quote/booking?",
    "For CID 7093495, what happened after /ccflownew/move-scope?",
    "Why did this customer end up on /ccflownew/quote/booking?",
]

---
### Strategy 1 — Semantic Retrieval Only

This is the plain dense retrieval baseline.

It is useful because it shows what happens when we rely only on vector similarity without any business-aware scoping.


In [3]:
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

query = TEST_QUERIES[0]
print(f"Query: {query}")
semantic_results = semantic_retriever.invoke(query)
show_results(semantic_results, "Semantic retrieval")

Query: For CID 7093495, what happened in this journey?

Semantic retrieval (5 chunks)
----------------------------------------------------------------------------------------------------
[1] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=route_entered | page=/ccflownew/isnj-journey
timestamp: 2026-04-02T06:51:49.308Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 32
action: route_entered
event_type: navigation
status: success
level: info
page_path: /ccflownew/isnj-journey
source: router
message: Route entered
result_json: {"pathname": "/ccflownew/isnj-jou

[2] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=route_entered | page=/ccflownew/date
timestamp: 2026-04-02T06:50:51.925Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 14
action: route_entered
event_typ

---
### Strategy 2 — Metadata-Scoped Retrieval

For debugging questions, semantic search improves a lot when we add business filters like `customer_id` and route filters like `page_path`.

This is not a different vector search algorithm. It is the same semantic retrieval, but scoped to the right slice of the logs.


In [4]:
query = TEST_QUERIES[1]
event_filter = build_scope_filter(query, chunk_type="event")

print(f"Query: {query}")
print("Extracted metadata filter:", event_filter)

scoped_event_results = vectorstore.similarity_search(
    query,
    k=5,
    filter=event_filter,
)
show_results(scoped_event_results, "Customer-scoped event retrieval")

Query: For CID 7093495, which UI condition evaluated to true on /ccflownew/quote/booking?
Extracted metadata filter: {'customer_id': '7093495', 'page_path': '/ccflownew/quote/booking', 'chunk_type': 'event'}

Customer-scoped event retrieval (5 chunks)
----------------------------------------------------------------------------------------------------
[1] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=evaluate_ui_condition | page=/ccflownew/quote/booking
timestamp: 2026-04-02T06:52:29.027Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 77
action: evaluate_ui_condition
event_type: ui_condition
status: success
level: info
page_path: /ccflownew/quote/booking
source: ScriptedForm
message: UI condition evaluated
condition: true == t

[2] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=evaluate_ui_condition | page=/ccflownew/quote/booki

In [5]:
query = TEST_QUERIES[0]
summary_filter = build_scope_filter(query, chunk_type="execution_summary")
event_filter = build_scope_filter(query, chunk_type="event")

print(f"Query: {query}")
print("Summary filter:", summary_filter)
print("Event filter:", event_filter)

summary_results = vectorstore.similarity_search(query, k=2, filter=summary_filter)
event_results = vectorstore.similarity_search(query, k=5, filter=event_filter)

show_results(summary_results, "Summary chunks")
show_results(event_results, "Event chunks")

Query: For CID 7093495, what happened in this journey?
Summary filter: {'customer_id': '7093495', 'chunk_type': 'execution_summary'}
Event filter: {'customer_id': '7093495', 'chunk_type': 'event'}

Summary chunks (1 chunks)
----------------------------------------------------------------------------------------------------
[1] type=execution_summary | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=None | page=None
group_key: cid:7093495 | exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
journey_id: ccflownew
customer_ids: ['7093495']
execution_ids: ['exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88']
event_count: 78
step_range: 1 -> 78
status_counts: {'success': 78}
top_actions: {'route_entered': 22, 'evaluate_ui_condition': 16, 'patch_context': 11, '


Event chunks (5 chunks)
----------------------------------------------------------------------------------------------------
[1] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-

---
### Strategy 3 — Hybrid Debug Retrieval

Here, "hybrid" means we combine:
- metadata scoping by business key
- semantic ranking inside that scope
- multiple chunk types for both context and evidence

This is a better fit for debugging than returning only events or only summaries.


In [6]:
def hybrid_debug_retrieve(query: str, *, k_summary: int = 1, k_events: int = 4):
    base_filter = build_scope_filter(query) or {}

    # Summary chunks usually do not carry page-level metadata, so keep them customer-scoped.
    summary_filter = {"chunk_type": "execution_summary"}
    if "customer_id" in base_filter:
        summary_filter["customer_id"] = base_filter["customer_id"]

    # Event chunks can be filtered more tightly, including page path when present.
    event_filter = {"chunk_type": "event", **base_filter}

    summary_docs = vectorstore.similarity_search(
        query, k=k_summary, filter=summary_filter
    )
    event_docs = vectorstore.similarity_search(query, k=k_events, filter=event_filter)
    return dedupe_docs(summary_docs + event_docs)


query = "For CID 7093495, why did this move end up on /ccflownew/quote/booking?"
print(f"Query: {query}")
hybrid_results = hybrid_debug_retrieve(query)
show_results(hybrid_results, "Hybrid summary + evidence retrieval")

Query: For CID 7093495, why did this move end up on /ccflownew/quote/booking?

Hybrid summary + evidence retrieval (5 chunks)
----------------------------------------------------------------------------------------------------
[1] type=execution_summary | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=None | page=None
group_key: cid:7093495 | exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
journey_id: ccflownew
customer_ids: ['7093495']
execution_ids: ['exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88']
event_count: 78
step_range: 1 -> 78
status_counts: {'success': 78}
top_actions: {'route_entered': 22, 'evaluate_ui_condition': 16, 'patch_context': 11, '

[2] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=route_entered | page=/ccflownew/quote/booking
timestamp: 2026-04-02T06:52:29.027Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 7

---
### Strategy 4 — Side-by-Side Comparison

This comparison is less about raw count and more about what kinds of chunks each strategy returns.

For debugging, a strong result set usually contains:
- one summary chunk for context
- several event chunks for proof


In [7]:
def chunk_mix(docs):
    return dict(Counter(doc.metadata.get("chunk_type", "?") for doc in docs))


for query in TEST_QUERIES:
    semantic_docs = semantic_retriever.invoke(query)

    scope_filter = build_scope_filter(query)
    if scope_filter:
        scoped_docs = vectorstore.similarity_search(query, k=5, filter=scope_filter)
    else:
        scoped_docs = semantic_docs

    hybrid_docs = hybrid_debug_retrieve(query)

    print("=" * 100)
    print(f"Query: {query}")
    print("Extracted scope:", scope_filter)
    print("Semantic mix:", chunk_mix(semantic_docs))
    print("Scoped mix:", chunk_mix(scoped_docs))
    print("Hybrid mix:", chunk_mix(hybrid_docs))
    print()

Query: For CID 7093495, what happened in this journey?
Extracted scope: {'customer_id': '7093495'}
Semantic mix: {'event': 5}
Scoped mix: {'event': 5}
Hybrid mix: {'execution_summary': 1, 'event': 4}

Query: For CID 7093495, which UI condition evaluated to true on /ccflownew/quote/booking?
Extracted scope: {'customer_id': '7093495', 'page_path': '/ccflownew/quote/booking'}
Semantic mix: {'event': 5}
Scoped mix: {'event': 5}
Hybrid mix: {'execution_summary': 1, 'event': 4}

Query: For CID 7093495, what happened after /ccflownew/move-scope?
Extracted scope: {'customer_id': '7093495', 'page_path': '/ccflownew/move-scope'}
Semantic mix: {'event': 5}
Scoped mix: {'event': 5}
Hybrid mix: {'execution_summary': 1, 'event': 4}

Query: Why did this customer end up on /ccflownew/quote/booking?
Extracted scope: {'page_path': '/ccflownew/quote/booking'}
Semantic mix: {'event': 5}
Scoped mix: {'event': 5}
Hybrid mix: {'execution_summary': 1, 'event': 4}



---
### Takeaways

| Strategy | What it does well | Main weakness | Best use |
|----------|-------------------|---------------|----------|
| Semantic only | Works with vague natural language | Can search too broadly | baseline behavior |
| Metadata scoped | Improves precision fast | Needs a business key like CID | customer-specific debugging |
| Hybrid debug retrieval | Gives both context and evidence | Slightly more logic | production RAG flow for logs |

**Recommended direction for the app:**
- parse the question for `customer_id` and exact page path when available
- retrieve both summary and event chunks
- rerank the combined evidence before answer generation

This sets up Notebook 4, where the LangGraph flow can turn user questions into scoped retrieval plus grounded answers.
